In [17]:
from sklearn.datasets import fetch_20newsgroups
from collections import Counter
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import spacy
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

1 - Baixar o corpus

In [18]:
colecao_medicina = fetch_20newsgroups(
    subset='train', 
    categories=['sci.med'], 
    remove=('headers', 'footers', 'quotes'))

colecao_query_medicina = fetch_20newsgroups(
    subset='test', 
    categories=['sci.med'], 
    remove=('headers', 'footers', 'quotes'))

2 - Fazer pré-processamento

In [19]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess_stemming(text):

    text = text.lower()

    text = re.sub(r"[^a-z\s]", " ", text)

    tokens = text.split()

    return " ".join(
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words
        and token.isalpha()
        and len(token) > 2
    )

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
def preprocess_lemma(texts):

    processed = []

    for doc in nlp.pipe(
        texts,
        batch_size=100,
        n_process=-1
    ):

        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
            and token.is_alpha
            and len(token) > 2
        ]

        processed.append(" ".join(tokens))

    return processed

In [20]:
print("Iniciando stemming...")
colecao_medicina.data = [preprocess_stemming(doc) for doc in colecao_medicina.data]
colecao_query_medicina.data = [preprocess_stemming(doc) for doc in colecao_query_medicina.data]
print("Stemming concluído.")
print(f"Exemplo de documento após stemming: {colecao_medicina.data[0][:200]}")
print("Iniciando lematização...")
colecao_medicina.data = [preprocess_lemma(doc) for doc in colecao_medicina.data]
colecao_query_medicina.data = [preprocess_lemma(doc) for doc in colecao_query_medicina.data]
print(f"Exemplo de documento após stemming: {colecao_medicina.data[0][:200]}")

Iniciando stemming...
Stemming concluído.
Exemplo de documento após stemming: repli keith actrix gen keith stewart would help anyon els ask medic inform subject could ask specif question one like type textbook chapter cover aspect subject look comprehens review ask local hospit
Iniciando lematização...
Exemplo de documento após stemming: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '',

Agora vamos iniciar o processo de modelagem. Vamos fazer de três formas: usando TF, TF-IDF, 

In [21]:
def build_representation(documents, method):

    if method == "tf":
        vectorizer = CountVectorizer(
            min_df=2,
            max_df=0.8
        )

    elif method == "tfidf":
        vectorizer = TfidfVectorizer(
            min_df=2,
            max_df=0.8
        )

    else:
        raise ValueError("Método inválido")

    X = vectorizer.fit_transform(documents)

    return X, vectorizer

In [22]:
X_tf, vec_tf = build_representation(
    colecao_medicina.data,
    method="tf"
)

X_tfidf, vec_tfidf = build_representation(
    colecao_medicina.data,
    method="tfidf"
)

Y_tf, vec_tf_query = build_representation(
    colecao_query_medicina.data,
    method="tf"
)
Y_tfidf, vec_tfidf_query = build_representation(    
    colecao_query_medicina.data,
    method="tfidf"
)

AttributeError: 'list' object has no attribute 'lower'

In [ ]:
print("Criando matriz TF-IDF...")
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None
)
X = vectorizer.fit_transform(colecao_medicina.data)
print(f"Matriz TF-IDF criada com shape: {X.shape}")

In [ ]:
print("===== REPRESENTAÇÃO VETORIAL =====")
print("Número de documentos:", X_tfidf.shape[0])
print("Tamanho do vocabulário:", X_tfidf.shape[1])
print("Número de elementos não nulos:", X_tfidf.nnz)

densidade = X_tfidf.nnz / (
    X_tfidf.shape[0] * X_tfidf.shape[1]
)

print("Densidade da matriz:", densidade)